In [3]:
!nvidia-smi

Thu Sep 17 12:06:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!nvidia-smi topo -m

	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks


In [ ]:
!pip install -q torch torchvision matplotlib

In [5]:
%%writefile model.py
"""Общая MLP модель для всех стратегий параллелизма."""

import torch
import torch.nn as nn


class MLP(nn.Module):
    """3-layer MLP для CIFAR-10: 3072 → 512 → 512 → 10."""

    def __init__(self, in_dim: int = 3072, hidden: int = 512, num_classes: int = 10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(x.size(0), -1)  # flatten
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

Writing model.py


In [6]:
%%writefile train_dp.py
"""DP: single-process data parallelism (master GPU bottleneck)."""

import time, json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from model import MLP, count_params


def main():
    device = torch.device("cuda:0")
    model = MLP().to(device)
    model = nn.DataParallel(model, device_ids=[0, 1])

    print(f"DP | params: {count_params(model.module) / 1e6:.2f}M")

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    train_ds = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=2)

    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    torch.cuda.reset_peak_memory_stats(device)
    start_time = time.time()
    total_samples = 0
    losses = []
    accs = []

    for epoch in range(10):
        model.train()
        correct = 0
        total = 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            total_samples += data.size(0)
            losses.append(loss.item())
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)

        acc = correct / total
        accs.append(acc)
        print(f"Epoch {epoch} | loss: {loss.item():.4f} | acc: {acc:.4f}")

    torch.cuda.synchronize()
    elapsed = time.time() - start_time
    peak_mem = torch.cuda.max_memory_allocated(device) / 1e9

    result = {
        "strategy": "DP",
        "model": "MLP",
        "params_m": count_params(model.module) / 1e6,
        "world_size": 2,
        "epochs": 10,
        "batch_size": 256,
        "total_time_s": elapsed,
        "throughput_samples_s": total_samples / elapsed,
        "peak_memory_gb": peak_mem,
        "final_loss": losses[-1],
        "final_acc": accs[-1],
        "losses": losses,
        "accs": accs,
    }
    with open("results/results_dp.json", "w") as f:
        json.dump(result, f)
    print(f"DP | time={elapsed:.1f}s | throughput={total_samples/elapsed:.1f} samples/s | peak_mem={peak_mem:.2f} GB | acc={accs[-1]:.4f}")


if __name__ == "__main__":
    main()

Writing train_dp.py


In [7]:
%%writefile train_ddp.py
"""DDP: multi-process data parallelism with NCCL all-reduce."""

import os, time, json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
from torchvision import datasets, transforms

from model import MLP, count_params


def main():
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    local_rank = int(os.environ["LOCAL_RANK"])

    dist.init_process_group(backend="nccl")
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")

    # Model
    model = MLP().to(device)
    model = DDP(model, device_ids=[local_rank])

    if rank == 0:
        print(f"DDP | params: {count_params(model.module) / 1e6:.2f}M")

    # Data
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    train_ds = datasets.CIFAR10("./data", train=True, download=True, transform=transform)
    sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(train_ds, batch_size=256, sampler=sampler, num_workers=2)

    optimizer = optim.AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    torch.cuda.reset_peak_memory_stats(device)
    start_time = time.time()
    total_samples = 0
    losses = []
    accs = []

    for epoch in range(10):
        sampler.set_epoch(epoch)
        model.train()
        correct = 0
        total = 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

            total_samples += data.size(0)
            losses.append(loss.item())
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)

        acc = correct / total
        accs.append(acc)
        if rank == 0:
            print(f"Epoch {epoch} | loss: {loss.item():.4f} | acc: {acc:.4f}")

    torch.cuda.synchronize()
    elapsed = time.time() - start_time

    if rank == 0:
        peak_mem = torch.cuda.max_memory_allocated(device) / 1e9
        result = {
            "strategy": "DDP",
            "model": "MLP",
            "params_m": count_params(model.module) / 1e6,
            "world_size": world_size,
            "epochs": 10,
            "batch_size": 256,
            "total_time_s": elapsed,
            "throughput_samples_s": total_samples / elapsed,
            "peak_memory_gb": peak_mem,
            "final_loss": losses[-1],
            "final_acc": accs[-1],
            "losses": losses,
            "accs": accs,
        }
        with open("results/results_ddp.json", "w") as f:
            json.dump(result, f)
        print(f"DDP | time={elapsed:.1f}s | throughput={total_samples/elapsed:.1f} samples/s | peak_mem={peak_mem:.2f} GB | acc={accs[-1]:.4f}")

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing train_ddp.py


In [8]:
%%writefile nccl_benchmark.py
"""NCCL communication benchmark: AllReduce, AllGather, ReduceScatter.

Uses nccl-tests binary if available, otherwise falls back to PyTorch.
"""

import os
import subprocess
import json
import re


def run_nccl_test(op: str, min_size: str = "1M", max_size: str = "256M",
                   gpus: int = 2, output_file: str = None):
    """Запускает nccl-tests для операции."""
    binary = f"./nccl-tests/build/{op}_perf"

    if not os.path.exists(binary):
        print(f"WARNING: {binary} not found. Compile nccl-tests first.")
        print("Run: git clone https://github.com/NVIDIA/nccl-tests.git")
        print("     cd nccl-tests && make MPI=0 CUDA_HOME=/usr/local/cuda")
        return None

    cmd = [binary, "-b", min_size, "-e", max_size, "-f", "2", "-g", str(gpus)]
    print(f"Running: {' '.join(cmd)}")

    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stdout

    # Парсим вывод
    # Формат: size count type redop root time algbw busbw #wrong ...
    lines = output.strip().split("\n")
    data = []
    for line in lines:
        parts = line.split()
        if len(parts) >= 8 and parts[0].isdigit():
            try:
                size_bytes = int(parts[0])
                busbw = float(parts[7])
                data.append({
                    "size_bytes": size_bytes,
                    "size_mb": size_bytes / 1e6,
                    "busbw_gbps": busbw,
                })
            except (ValueError, IndexError):
                continue

    result_dict = {
        "operation": op,
        "gpus": gpus,
        "data": data,
    }

    if output_file:
        with open(output_file, "w") as f:
            json.dump(result_dict, f, indent=2)

    return result_dict


def main():
    os.makedirs("results", exist_ok=True)

    # AllReduce (DDP gradient sync)
    ar = run_nccl_test("all_reduce", output_file="results/nccl_all_reduce.json")

    # AllGather (ZeRO parameter collection)
    ag = run_nccl_test("all_gather", output_file="results/nccl_all_gather.json")

    # ReduceScatter (ZeRO gradient sharding)
    rs = run_nccl_test("reduce_scatter", output_file="results/nccl_reduce_scatter.json")

    # Печатаем summary
    print("\n=== NCCL Benchmark Summary ===")
    for name, data in [("AllReduce", ar), ("AllGather", ag), ("ReduceScatter", rs)]:
        if data and data["data"]:
            max_bw = max(d["busbw_gbps"] for d in data["data"])
            print(f"{name}: max busbw = {max_bw:.2f} GB/s")


if __name__ == "__main__":
    main()

Writing nccl_benchmark.py


In [9]:
%%writefile topology_analysis.py
"""Анализ топологии GPU: PCIe vs NVLink."""

import subprocess
import json


def get_topology():
    """Запускает nvidia-smi topo -m и парсит вывод."""
    result = subprocess.run(
        ["nvidia-smi", "topo", "-m"],
        capture_output=True, text=True,
    )
    return result.stdout


def get_gpu_info():
    """Получает информацию о GPU."""
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
         "--format=csv,noheader"],
        capture_output=True, text=True,
    )
    return result.stdout.strip()


def main():
    print("=== GPU Info ===")
    print(get_gpu_info())

    print("\n=== Topology ===")
    topo = get_topology()
    print(topo)

    # Анализ
    print("\n=== Analysis ===")
    if "PHB" in topo:
        print("Connection type: PHB (PCIe Host Bridge)")
        print("Bandwidth: ~16 GB/s (PCIe 3.0 x16)")
    if "NV" in topo and "NV1" in topo:
        print("NVLink detected: ~300 GB/s (A100)")
    elif "PHB" in topo:
        print("No NVLink. Communication goes through PCIe.")
        print("This limits bandwidth to ~16 GB/s.")

    # Сохраняем в JSON
    result = {
        "gpu_info": get_gpu_info(),
        "topology": topo,
    }
    with open("results/topology.json", "w") as f:
        json.dump(result, f, indent=2)


if __name__ == "__main__":
    main()

Writing topology_analysis.py


In [10]:
%%writefile gloo_vs_nccl.py
"""Сравнение Gloo vs NCCL на all_reduce."""

import os
import time
import json
import torch
import torch.distributed as dist
import torch.multiprocessing as mp


def benchmark_backend(backend: str, rank: int, world_size: int,
                       size_mb: int = 64, num_iters: int = 50):
    """Бенчмарк all_reduce для заданного backend."""
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29510"

    dist.init_process_group(backend=backend, rank=rank, world_size=world_size)

    if backend == "nccl":
        torch.cuda.set_device(rank)
        device = f"cuda:{rank}"
    else:
        device = "cpu"

    num_elements = size_mb * 1024 * 1024 // 4
    tensor = torch.randn(num_elements, device=device)

    for _ in range(5):
        dist.all_reduce(tensor)

    if backend == "nccl":
        torch.cuda.synchronize()

    start = time.time()
    for _ in range(num_iters):
        dist.all_reduce(tensor)
    if backend == "nccl":
        torch.cuda.synchronize()
    elapsed = time.time() - start

    busbw = 2 * (world_size - 1) / world_size * tensor.nbytes * num_iters / elapsed / 1e9

    dist.destroy_process_group()
    return busbw


# === TOP-LEVEL WORKERS (не внутри main) ===

def nccl_worker(rank, world_size, size_mb, return_dict):
    bw = benchmark_backend("nccl", rank, world_size, size_mb)
    if rank == 0:
        return_dict["nccl"] = bw


def gloo_worker(rank, world_size, size_mb, return_dict):
    bw = benchmark_backend("gloo", rank, world_size, size_mb)
    if rank == 0:
        return_dict["gloo"] = bw


def main():
    results = {}
    size_mb = 64

    # NCCL
    if torch.cuda.is_available():
        world_size = torch.cuda.device_count()
        manager = mp.Manager()
        return_dict = manager.dict()
        mp.spawn(nccl_worker, args=(world_size, size_mb, return_dict),
                 nprocs=world_size, join=True)
        results["nccl"] = return_dict.get("nccl", 0)
        print(f"NCCL all_reduce ({size_mb} MB): {results['nccl']:.2f} GB/s")

    # Gloo (CPU)
    manager = mp.Manager()
    return_dict = manager.dict()
    mp.spawn(gloo_worker, args=(2, size_mb, return_dict), nprocs=2, join=True)
    results["gloo"] = return_dict.get("gloo", 0)
    print(f"Gloo all_reduce ({size_mb} MB): {results['gloo']:.2f} GB/s")

    with open("results/gloo_vs_nccl.json", "w") as f:
        json.dump(results, f, indent=2)


if __name__ == "__main__":
    main()

Overwriting gloo_vs_nccl.py


In [11]:
%%writefile plot_results.py
"""Все графики: DP vs DDP, NCCL bandwidth, topology."""

import json
import os
import matplotlib.pyplot as plt
import numpy as np


def load(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def main():
    os.makedirs("results/figures", exist_ok=True)

    dp = load("results/results_dp.json")
    ddp = load("results/results_ddp.json")
    ar = load("results/nccl_all_reduce.json")
    ag = load("results/nccl_all_gather.json")
    rs = load("results/nccl_reduce_scatter.json")
    gn = load("results/gloo_vs_nccl.json")

    fig = plt.figure(figsize=(20, 12))

    # 1. DP vs DDP throughput
    ax1 = fig.add_subplot(2, 3, 1)
    if dp and ddp:
        strategies = ["DP", "DDP"]
        throughputs = [dp["throughput_samples_s"], ddp["throughput_samples_s"]]
        bars = ax1.bar(strategies, throughputs,
                        color=["#3498db", "#2ecc71"], edgecolor="black")
        ax1.set_ylabel("Throughput (samples/s)")
        ax1.set_title("DP vs DDP: Throughput")
        for bar, v in zip(bars, throughputs):
            ax1.text(bar.get_x() + bar.get_width()/2, v + 50,
                     f"{v:.0f}", ha="center", fontsize=10)

    # 2. DP vs DDP time
    ax2 = fig.add_subplot(2, 3, 2)
    if dp and ddp:
        times = [dp["total_time_s"], ddp["total_time_s"]]
        bars = ax2.bar(strategies, times,
                        color=["#3498db", "#2ecc71"], edgecolor="black")
        ax2.set_ylabel("Training time (s)")
        ax2.set_title("DP vs DDP: Time (3 epochs)")
        for bar, v in zip(bars, times):
            ax2.text(bar.get_x() + bar.get_width()/2, v + 0.3,
                     f"{v:.1f}s", ha="center", fontsize=10)

    # 3. DP vs DDP accuracy
    ax3 = fig.add_subplot(2, 3, 3)
    if dp and ddp:
        accs = [dp["final_acc"], ddp["final_acc"]]
        bars = ax3.bar(strategies, accs,
                        color=["#3498db", "#2ecc71"], edgecolor="black")
        ax3.set_ylabel("Accuracy")
        ax3.set_title("DP vs DDP: Accuracy")
        ax3.set_ylim(0, 0.7)
        for bar, v in zip(bars, accs):
            ax3.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                     f"{v:.3f}", ha="center", fontsize=10)

    # 4. NCCL AllReduce bandwidth
    ax4 = fig.add_subplot(2, 3, 4)
    if ar and ar["data"]:
        sizes = [d["size_mb"] for d in ar["data"]]
        bw = [d["busbw_gbps"] for d in ar["data"]]
        ax4.plot(sizes, bw, 'o-', color='#e74c3c', linewidth=2, markersize=8)
        ax4.set_xlabel("Message size (MB)")
        ax4.set_ylabel("Bus bandwidth (GB/s)")
        ax4.set_title("NCCL AllReduce on 2×T4")
        ax4.set_xscale('log', base=2)
        ax4.grid(True, alpha=0.3)
        ax4.axhline(y=16, color='gray', linestyle='--', label='PCIe 3.0 x16 limit')
        ax4.legend()

    # 5. AllGather vs ReduceScatter
    ax5 = fig.add_subplot(2, 3, 5)
    if ag and ag["data"]:
        sizes = [d["size_mb"] for d in ag["data"]]
        bw = [d["busbw_gbps"] for d in ag["data"]]
        ax5.plot(sizes, bw, 's-', color='#3498db', linewidth=2, label='AllGather')
    if rs and rs["data"]:
        sizes = [d["size_mb"] for d in rs["data"]]
        bw = [d["busbw_gbps"] for d in rs["data"]]
        ax5.plot(sizes, bw, '^-', color='#9b59b6', linewidth=2, label='ReduceScatter')
    ax5.set_xlabel("Message size (MB)")
    ax5.set_ylabel("Bus bandwidth (GB/s)")
    ax5.set_title("AllGather vs ReduceScatter")
    ax5.set_xscale('log', base=2)
    ax5.legend()
    ax5.grid(True, alpha=0.3)

    # 6. Gloo vs NCCL
    ax6 = fig.add_subplot(2, 3, 6)
    if gn:
        backends = ["Gloo (CPU)", "NCCL (GPU)"]
        bws = [gn.get("gloo", 0), gn.get("nccl", 0)]
        bars = ax6.bar(backends, bws,
                        color=["#95a5a6", "#e74c3c"], edgecolor="black")
        ax6.set_ylabel("Bus bandwidth (GB/s)")
        ax6.set_title("Gloo vs NCCL (64 MB AllReduce)")
        for bar, v in zip(bars, bws):
            ax6.text(bar.get_x() + bar.get_width()/2, v + 0.1,
                     f"{v:.2f}", ha="center", fontsize=10)

    plt.suptitle("Distributed Training + NCCL Benchmark on 2×T4 (Kaggle)",
                 fontsize=16)
    plt.tight_layout()
    plt.savefig("results/figures/full_benchmark.png", dpi=150)
    print("Saved: results/figures/full_benchmark.png")


if __name__ == "__main__":
    main()

Writing plot_results.py


In [12]:
# 1. NCCL tests (компиляция)
!git clone https://github.com/NVIDIA/nccl-tests.git
!cd nccl-tests && make MPI=0 CUDA_HOME=/usr/local/cuda -j4
!mkdir -p results

# 2. NCCL benchmarks
!python nccl_benchmark.py

# 3. Topology
!python topology_analysis.py

# 4. Gloo vs NCCL
!python gloo_vs_nccl.py

# 5. DP/DDP
!python train_dp.py
!torchrun --standalone --nproc_per_node=2 train_ddp.py

# 6. Графики
!python plot_results.py

Cloning into 'nccl-tests'...
remote: Enumerating objects: 1360, done.
remote: Counting objects: 100% (771/771), done.
remote: Compressing objects: 100% (262/262), done.
remote: Total 1360 (delta 720), reused 509 (delta 509), pack-reused 589 (from 4)
Receiving objects: 100% (1360/1360), 443.19 KiB | 2.74 MiB/s, done.
Resolving deltas: 100% (916/916), done.
make -C src build BUILDDIR=/kaggle/working/nccl-tests/build
make[1]: Entering directory '/kaggle/working/nccl-tests/src'
make[2]: Entering directory '/kaggle/working/nccl-tests/os'
Compiling  timer.cc                            > /kaggle/working/nccl-tests/build/timer.o
Compiling /kaggle/working/nccl-tests/build/verifiable/verifiable.o
nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Compiling  all_reduce.cu                       > /kaggle/working/nccl-tests/build/all_reduce.o
nvcc warning : Support